检查Pytorch

In [2]:
import torch
print("PyTorch version:",torch.__version__)

PyTorch version: 2.14.0+cu132


## 1. Tensor基础
Tensor可以理解为PyTorch中的多维数组

常见属性：
- shape：张量形状
- dtype：元素的数据类型
- device：张量所在设备
- requires_grad：是否需要追踪梯度

In [3]:
x = torch.tensor([
    [1,2,3],
    [4,5,6]
])

print(x)
print("shape:",x.shape)
print("dtype:",x.dtype)
print("device:",x.device)

tensor([[1, 2, 3],
        [4, 5, 6]])
shape: torch.Size([2, 3])
dtype: torch.int64
device: cpu


以上代码中：
torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])全部是整数，所以通常得到：torch.int64  
而在创建tensor时可以指定dtype

In [4]:
x = torch.tensor([
    [1,2,3],
    [4,5,6]
],
dtype = torch.float32)
print(x)
print(x.dtype)

tensor([[1., 2., 3.],
        [4., 5., 6.]])
torch.float32


dtype很重要，因为dtype不同，**Tensor中每个元素的数据类型不同**  
而以后模型参数、输入数据、GPU算子都可能对dtype有要求

In [6]:
x = torch.tensor(
    [1.0, 2.0, 3.0],
    device = "cpu"  #此处device表示：Tensor的数据实际放在哪里计算
)
print(x)
print(x.device)

tensor([1., 2., 3.])
cpu


In [7]:
print("CUDA available:",torch.cuda.is_available())

CUDA available: True


In [8]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

x = torch.tensor(
    [1.0, 2.0, 3.0],
    device = device
)

print(x)
print(x.device)

tensor([1., 2., 3.], device='cuda:0')
cuda:0


记住：
Tensor  
├── 数据值  
├── shape  
├── dtype  
└── device

## 2. reshape 与 permute
（1）reshape:  
改变Tensor的形状，但元素总数必须保持一致

In [13]:
x = torch.arange(12)

print(x)
print("shape:",x.shape)

y = x.reshape(3,4)
print(y)
print(y.shape)

z = x.reshape(2,6)
print(z)
print(z.shape)

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
shape: torch.Size([12])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
torch.Size([3, 4])
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
torch.Size([2, 6])


python中reshape默认为行优先，因此先把一行填满再到下一行  
对于reshape，应有：  
**reshape 前元素数量 = reshape 后元素数量**

PyTorch可以帮你自动推断一个维度，使用 **-1来表示需要被自动推断的维度**

In [15]:
x = torch.arange(12)
y = x.reshape(3,-1)
print(y)
print(y.shape)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
torch.Size([3, 4])


（2）下述permute  
reshape和permute非常容易混淆，它们之间的核心区别可以记为：  
reshape→ 重新组织 shape  permute→ 调换维度顺序  
原本tensor的维度排列数序为(0,1,2)，进行permute(2,0,1)后，就将原来的第2维改为现在的第0维，依此类推  
permute 对以后语音、图像、Transformer 非常重要。  
例如：(batch, time, feature) 有时候某个模块要求：(time, batch, feature) 就可以用permute(1,0,2)

In [16]:
x = torch.arange(24).reshape(2,3,4)
print(x.shape)

y = x.permute(2,0,1)
print(y.shape)

torch.Size([2, 3, 4])
torch.Size([4, 2, 3])


In [17]:
#一个简单的二位permute
x = torch.tensor([
    [1,2,3],
    [4,5,6]
])
print(x)
print(x.shape)

y = x.permute(1,0)
print(y)
print(y.shape)

tensor([[1, 2, 3],
        [4, 5, 6]])
torch.Size([2, 3])
tensor([[1, 4],
        [2, 5],
        [3, 6]])
torch.Size([3, 2])


## 3. Autograd 与计算图

当Tensor设置 requires_grad = True后，PyTorch会记录相关运算，从而自动计算梯度。  
基本流程：  
x  
↓  
前向计算  
↓  
y  
↓  
y.backward()  
↓  
x.grad  
PyTorch 的 autograd 会在前向计算时记录需要的操作，并在 backward() 时利用链式法则计算梯度；需要梯度的叶子 Tensor 会把结果累积到 .grad。

In [30]:
x = torch.tensor(
    2.0,
    requires_grad = True
)
print(x)
print("requires_grad:",x.requires_grad)
print("grad:",x.grad)

tensor(2., requires_grad=True)
requires_grad: True
grad: None


现在的x.grad = None，因为还没有进行backward()

In [31]:
x = torch.tensor(2.0,requires_grad=True)

y = x**3

print("x =",x)
print("y =",y)

print("x.requires_grad:",x.requires_grad)
print("y.requires_grad:",y.requires_grad)

print("x.grad_fn:",x.grad_fn)
print("y.grad_fn:",y.grad_fn)

x = tensor(2., requires_grad=True)
y = tensor(8., grad_fn=<PowBackward0>)
x.requires_grad: True
y.requires_grad: True
x.grad_fn: None
y.grad_fn: <PowBackward0 object at 0x0000026E0B837B80>


这段代码演示的是 PyTorch 的自动求导机制 autograd，核心是 requires_grad、计算图和 grad_fn。  
x 是用户直接创建的，所以它是叶子张量 leaf tensor。叶子张量的 grad_fn 通常是 None，因为它不是由某个运算产生的，而是计算图的起点。  
对于y=x**3  
因为x.requires_grad=True，所以这个运算会被记录到计算图中  
PyTorch 会记录：y 是由 x 经过幂运算得到的，因此:  
- y.requires_grad自动变为True
- y.grad_fn指向对应的反向函数，通常是PowBackward0
y是运算结果，不是用户直接创建的，所以它是**非叶子张量**  
#### 关键概念
- grad_fn：表示这个张量是由什么运算产生的，以及反向传播时该用哪个反向函数。他不是梯度值。
- x.grad_fn is None：因为x是叶子节点，是计算图起点
- y.grad_fn是PowBackward0，因为y=x**3，反向传播时要用幂函数的反向规则
- 实际梯度值不在grad_fn里，而是在.grad属性里，此时x.grad还是None


In [32]:
print("x.is_leaf:",x.is_leaf)
print("y.is_leaf:",y.is_leaf)

x.is_leaf: True
y.is_leaf: False


#### Autograd验证

In [33]:
x = torch.tensor(2.0,requires_grad=True)
y = x**3
y.backward()    #可以理解为：从y开始，沿计算图反向计算梯度
print("x =",x)
print("y =",y)
print("x.grad =",x.grad)

x = tensor(2., requires_grad=True)
y = tensor(8., grad_fn=<PowBackward0>)
x.grad = tensor(12.)


y.backward()可以理解为：从y开始，沿计算图反向计算梯度，而x.grad就是\(\frac{\partial y}{\partial x}\)

#### 计算图到底干了什么
可以理解为在进行Tensor计算时，自动创建计算图，然后用backward()可以得到所需grad，并释放计算时建立的计算图。
前向：  
x = 2  
│  
│ x³  
↓  
y = 8  
反向：  
x = 2  
↑  
│ dy/dx = 3x² = 12  
│  
y = 8  
所以：  
forward  
→ 算数值  
backward  
→ 算梯度  

In [34]:
x = torch.tensor(2.0,requires_grad=True)
y = x**3
y.backward()   
print("x.grad =",x.grad)
y.backward()

x.grad = tensor(12.)


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

如果确实需要在同一个计算图上执行多次 backward，需要保留该图；**正常情况下则不要直接重复使用已经反传过的旧图**  
**为什么第二次backward会失败**  
- 第一次：y.backward()完成后，为了节省内存，PyTorch默认会释放反向传播需要保存的中间信息。 
- 第二次：y.backward()相当于：想拿已经释放了关键中间信息的旧图再算一次。因此报错
y.backward(retain_graph=True)可以要求PyTorch保留图，但是正常训练更常见：  
Forward -> 产生新计算图   Backward -> 旧图用完   下一轮Forward -> 重新产生新计算图  

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y1 = x**3
y1.backward()
print("第一次backward后：")
print(x.grad)

y2 = x**3
y2.backward()
print("第二次backward后：")
print(x.grad)

第一次backward后：
tensor(12.)
第二次backward后：
tensor(24.)


**第一次backward后重新构建计算图然后再次backward发现梯度变为1、2次之和**  
**原因** ： Pytorch默认不是覆盖，而是累加，PyTorch 官方文档明确说明：backward() 会把新的梯度累积进叶子 Tensor 已有的 .grad 中，而不是默认覆盖。

**为什么深度学习要梯度累加？**  
以后可能会主动做：gradient accumulation  
例如显存只能一次放：batch = 4，但是想要模拟：batch = 16  
可以多次backward累加梯度，再更新参数

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y1 = x**3
y1.backward()
print(x.grad)

x.grad.zero_()
print(x.grad)

y2 = x ** 3
y2.backward()
print(x.grad)

tensor(12.)
tensor(0.)
tensor(12.)


**zero_grad到底是什么？**  
在真正训练模型时，一般不是:x.grad.zero_()，而是：optimizer.zero_grad()  
典型训练循环：  
optimizer.zero_grad()  
output = model(x)  
loss = loss_fn(output, target)  
loss.backward()  
optimizer.step()  
现在先理解这个逻辑：  
上一轮梯度  
   ↓  
zero_grad()  
清掉  
   ↓
forward  
   ↓  
loss  
   ↓  
backward()  
算这一轮梯度  
   ↓  
optimizer.step()  
更新参数  
默认情况下每次 .backward() 都会累积梯度，因此训练时通常需要在合适的位置清理梯度。

#### detach

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**3
print(y)
print("requires_grad:", y.requires_grad)
print("grad_fn:", y.grad_fn)

tensor(8., grad_fn=<PowBackward0>)
requires_grad: True
grad_fn: <PowBackward0 object at 0x0000026E0B837B20>


In [ ]:
z = y.detach()
print(z)
print("requires_grad:", z.requires_grad)
print("grad_fn:", z.grad_fn)

tensor(8.)
requires_grad: False
grad_fn: None


**detach是什么意思？**  
通过以上代码输出结果可以看出：在取y.detach()后z变为仅有数值的tensor，**官方 API 将 detach() 定义为返回一个与当前计算图分离的新 Tensor。**  
原来：x->y y仍然连接计算图  
而z=y.detach()，可以理解为：  
detach = 保留数值 + 切断与当前 autograd 图的连接

**什么时候用detach？**  
以后会有：loss.detach() 或者 features = features.detach()  
目的可能是：后续只想使用这个Tensor的数值，不希望梯度继续沿之前的计算图传播
例如模型 A：  
input  
  ↓  
Model A  
  ↓  
feature  
若：feature.detach()，那么后面的计算不会再通过这条detached Tensor往Model A传播梯度

In [ ]:
#另一个常见的写法
x = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
    y = x**3
print(y)
print("requires_grad:", y.requires_grad)
print("grad_fn:", y.grad_fn)

tensor(8.)
requires_grad: False
grad_fn: None


这里虽然：x.requires_grad == True，但是：with torch.no_grad():中的计算不会被记录进反向图。  
PyTorch 官方文档说明，no_grad 模式里的操作不会记录到 backward graph，即使输入 Tensor 原本 requires_grad=True。

**detach和no_grad的区别**  
| | `detach()` | `torch.no_grad()` |
|---|---|---|
| 对象 | 一个 Tensor | 一段代码 |
| 作用 | 把 Tensor 从已有计算图断开 | 这一代码块内不建立需要反传的图 |
| 保留数值 | 是 | 是 |
| 常见用途 | 断开某个中间结果 | 推理、参数更新等不需要梯度的运算 |

## 4. W1D4综合实验

In [36]:
import torch

print("=== Tensor basics ===")

x = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype = torch.float32
)
print("x:",x)
print("shape:",x.shape)
print("dtype:",x.dtype)
print("device:",x.device)

print("\n=== Reshape ===")
a = torch.arange(12)
print(a)
b = a.reshape(3,4)
print(b)
print("shape:",b.shape)

print("\n=== Permute ===")
c = torch.arange(24).reshape(2,3,4)

d = c.permute(2,0,1)
print("before:",c.shape)
print("after:",d.shape)

print("\n=== Autograd ===")
x = torch.tensor(2.0, requires_grad=True)
y = x**3
y.backward()

print("y:",y.item())
print("dy/dx:",x.grad.item())

print("\n=== Gradient accumulation ===")
y2 = x**3
y2.backward()

print("grad after second backward:",x.grad.item())
x.grad.zero_()
print("grad:",x.grad.item())

print("\n=== Detach ===")
y3 = x**3
z = y3.detach()
print("y3 requires_grad:", y3.requires_grad)
print("z requires_grad:", z.requires_grad)

print("\n=== no_grad ===")
with torch.no_grad():
    y4 = x**3
print("y4 requires_grad:", y4.requires_grad)

=== Tensor basics ===
x: tensor([1., 2., 3.])
shape: torch.Size([3])
dtype: torch.float32
device: cpu

=== Reshape ===
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
shape: torch.Size([3, 4])

=== Permute ===
before: torch.Size([2, 3, 4])
after: torch.Size([4, 2, 3])

=== Autograd ===
y: 8.0
dy/dx: 12.0

=== Gradient accumulation ===
grad after second backward: 24.0
grad: 0.0

=== Detach ===
y3 requires_grad: True
z requires_grad: False

=== no_grad ===
y4 requires_grad: False


**今天建立的完整思维模型**  
Tensor  
│  
├── shape  
│  
├── dtype  
│  
├── device  
│  
└── requires_grad  
        │  
        │ True  
        ↓  
执行运算  
        │  
        ↓  
构建计算图  
        │  
        ↓  
产生 output  
        │  
        ↓  
output.backward()  
        │  
        ↓  
沿计算图反向传播  
        │  
        ↓  
叶子 Tensor 的 .grad  

.backward()  
并不是把梯度“返回”出来  

而是通常把梯度累积到  
需要梯度的叶子 Tensor 的 .grad 中  